# Week 5 — Chất lượng mô hình và bằng chứng nghiên cứu

## Đọc notebook theo thứ tự
1. Quality gate: vì sao artifact cũ FAIL và phải retrain trên cloud.
2. Common mask, coverage và paired BS-cluster bootstrap.
3. A/B ablation, bảng candidate và error atlas.
4. Code thật của các module chính; ví dụ debug mask.

**Run All chỉ review, không training, không mở holdout, không freeze champion.**
Notebook lấy kết quả thật từ hosted validation worker đã hoàn tất.
Source dùng Python đọc được trực tiếp, có comment; không có encoded source blob.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import sys
import numpy as np
import pandas as pd
import yaml
from IPython.display import display, Markdown, Image

# Input là output của worker đã chạy xong; chỉ đọc kết quả, không retrain.
matches=list(Path('/kaggle/input').rglob('week5_comparison_manifest.json'))
if not matches:
    raise FileNotFoundError('Attach completed Week 5 validation notebook outputs as Kaggle Input.')
REPORTS=matches[0].parent
SOURCE=REPORTS.parent
WORK=Path('/kaggle/working/week5-review')
WORK.mkdir(exist_ok=True)
for folder in ('src','scripts','configs'):
    shutil.copytree(SOURCE/folder,WORK/folder,dirs_exist_ok=True)
os.chdir(WORK)
sys.path.insert(0,str(WORK))
RUN_TRAINING=False  # review notebook không có đường thực thi training
print('Reading completed reports:',REPORTS)


## 1. Quality gate và thay đổi bắt buộc

Artifact cũ dùng LightGBM44 và GRU static6, không đạt plan mới.
Đã retrain **6 LightGBM40** và **4 GRU phase4** trên cloud; không chỉ sửa inference.
Imputer/scaler là global per-KPI: A có 1 phần tử, B có 4; DLPRB statistics chung.
Beam summary chỉ dùng cho sampling/load-tier diagnostics, không đi vào model input.


In [ ]:
gate=json.loads((REPORTS/'model_quality_gate.json').read_text())
print('Quality gate:',gate['status'])
display(pd.DataFrame(gate['checks']))


## 2. Benchmark công bằng và CI

Mỗi horizon dùng đúng cùng `(origin, beam)` và actual cho mọi candidate.
Common mask chỉ yêu cầu actual/forecasts hữu hạn, không loại observation vì history có NaN.
Bootstrap resample nguyên BS cluster; 1.000 draws, seed17, 95% percentile CI.
CI cho MAE(A)−MAE(B): âm nghiêng về A. Có 30 BS clusters, nên diễn giải thận trọng.


In [ ]:
comparison=pd.read_csv(REPORTS/'model_comparison_val.csv')
columns=['model','horizon','mae','n_origins','n_pairs','n_beams','coverage_pct','samples_excluded_pct']
display(comparison[comparison.load_tier=='All'][columns])
paired=pd.read_csv(REPORTS/'paired_model_comparison_val.csv')
display(paired[(paired.model_a=='lightgbm_c1') & paired.model_b.isin(['lightgbm_c2','gru_A','gru_B'])])


## 3. A/B ablation và candidate-ready

Không chọn seed đẹp nhất. GRU family candidate là trung bình forecast của cả seeds17/42;
mean ± SD theo seed trong ablation chỉ mang tính mô tả.
Quy tắc: mean normalized MAE, ưu tiên family rẻ hơn khi paired CI không phân biệt.
Baseline được phép làm champion candidate. Đây chưa phải quyết định freeze cuối.


In [ ]:
display(pd.read_csv(REPORTS/'ablation_study.csv'))
display(pd.read_csv(REPORTS/'champion_candidate_table.csv'))
candidate=yaml.safe_load((SOURCE/'configs/champion_candidate.yaml').read_text())
print('Candidate:',candidate['family'],'| frozen:',candidate['frozen'],'| holdout:',candidate['holdout_allowed'])


## 4. Error atlas theo quy tắc, không cherry-pick

Top 5 beam theo mean horizon MAE; 5 đại diện mỗi tier bằng các rank cách đều trên fit mean.
Ba case: 2 peak underprediction lớn nhất và 1 zero-to-active miss lớn nhất tại h=24.
Cùng target hours/window cho LightGBM, GRU và best baseline mỗi horizon.
Đây là lỗi forecast retrospective, không phải nhãn congestion/incident.


In [ ]:
display(pd.read_csv(REPORTS/'error_atlas_beams.csv'))
display(pd.read_csv(REPORTS/'error_atlas_cases.csv'))
for name in ('actual_vs_forecast.png','mae_by_horizon.png','mae_by_load_tier.png'):
    display(Image(filename=str(REPORTS/'figures'/name)))


## 5. Code để review và debug

Mỗi cell bên dưới ghi đúng module được hiển thị; không chạy training.


### `src/evaluation/common_mask.py`

Khóa common finite pairs và phân biệt raw missingness với comparison exclusions.


In [ ]:
%%writefile src/evaluation/common_mask.py
"""Lock exactly the same finite observation pairs across a comparison set."""

import numpy as np


def comparable_mask(y_true: np.ndarray, predictions: dict[str, np.ndarray]) -> np.ndarray:
    """Require finite actual and every candidate forecast; history NaNs do not exclude pairs."""
    actual = np.asarray(y_true)
    if actual.ndim != 2 or not predictions:
        raise ValueError("Expected (origin,beam) actuals and at least one candidate")
    mask = np.isfinite(actual)
    for name, prediction in predictions.items():
        prediction = np.asarray(prediction)
        if prediction.shape != actual.shape:
            raise ValueError(f"Candidate shape differs: {name}")
        mask &= np.isfinite(prediction)
    return mask


def coverage_report(actual: np.ndarray, prediction: np.ndarray, mask: np.ndarray) -> dict:
    """Separate raw missingness from exclusions caused by the shared comparison set."""
    if actual.shape != prediction.shape or mask.shape != actual.shape:
        raise ValueError("Coverage arrays must have identical shapes")
    individual = np.isfinite(actual) & np.isfinite(prediction)
    if np.any(mask & ~individual):
        raise ValueError("Common mask includes an unavailable pair")
    return {
        "n_origins": int(mask.any(1).sum()),
        "n_pairs": int(mask.sum()),
        "n_beams": int(mask.any(0).sum()),
        "n_total_beams": actual.shape[1],
        "coverage_pct": float(mask.mean() * 100) if mask.size else 0.0,
        "samples_excluded_pct": float((1 - mask.mean()) * 100) if mask.size else 100.0,
        "candidate_coverage_pct": float(individual.mean() * 100) if mask.size else 0.0,
        "n_missing_target": int((~np.isfinite(actual)).sum()),
        "n_missing_prediction": int((~np.isfinite(prediction)).sum()),
        "n_excluded_due_to_comparison": int((individual & ~mask).sum()),
    }


### `src/evaluation/bootstrap.py`

Resample nguyên cluster và giữ cùng draws qua horizons.


In [ ]:
%%writefile src/evaluation/bootstrap.py
"""Paired cluster bootstrap; resample whole verified BS clusters, preserving paired errors."""

import numpy as np


def _stats(actual, a, b, clusters, mask):
    if actual.shape != a.shape or actual.shape != b.shape or mask.shape != actual.shape:
        raise ValueError("All paired arrays must have matching shapes")
    labels = np.unique(clusters)
    if len(clusters) != actual.shape[1]:
        raise ValueError("One cluster label per beam is required")
    if np.any(mask & ~(np.isfinite(actual) & np.isfinite(a) & np.isfinite(b))):
        raise ValueError("Bootstrap mask includes a nonfinite comparison pair")
    sums, counts = [], []
    delta = np.abs(a - actual) - np.abs(b - actual)
    for label in labels:
        selected = (clusters == label)[None, :] & mask
        sums.append(float(np.where(selected, delta, 0).sum()))
        counts.append(int(selected.sum()))
    return np.asarray(sums), np.asarray(counts)


def paired_multihorizon_ci(
    pairs: list[tuple],
    clusters: np.ndarray,
    scales: list[float],
    repetitions: int = 1000,
    seed: int = 17,
    confidence: float = 0.95,
) -> dict:
    """Mean normalized horizon MAE difference; use identical BS draws across horizons."""
    if not pairs or len(pairs) != len(scales) or repetitions < 2 or not 0 < confidence < 1:
        raise ValueError("Invalid paired bootstrap configuration")
    if any(not np.isfinite(s) or s <= 0 for s in scales):
        raise ValueError("Normalization scales must be positive")
    clusters = np.asarray(clusters)
    summaries = [_stats(pair[0], pair[1], pair[2], clusters, pair[3]) for pair in pairs]
    sums = np.stack([s / scales[i] for i, (s, _) in enumerate(summaries)])
    counts = np.stack([c for _, c in summaries])
    used = counts.sum(0) > 0
    sums, counts = sums[:, used], counts[:, used]
    if not counts.size or np.any(counts.sum(1) == 0) or used.sum() < 2:
        return {
            "difference": float("nan"),
            "lower": float("nan"),
            "upper": float("nan"),
            "status": "insufficient_clusters",
            "n_clusters": int(used.sum()),
        }
    n = int(used.sum())
    draws = np.random.default_rng(seed).multinomial(n, np.full(n, 1 / n), repetitions)
    denominator = counts @ draws.T
    valid = (denominator > 0).all(0)
    if valid.sum() < 2:
        return {"difference":float((sums.sum(1)/counts.sum(1)).mean()),
                "lower":float("nan"),"upper":float("nan"),
                "status":"insufficient_resamples","n_clusters":n}
    bootstrap = ((sums @ draws[valid].T) / denominator[:, valid]).mean(0)
    alpha = (1 - confidence) / 2
    low, high = np.quantile(bootstrap, [alpha, 1 - alpha])
    return {
        "difference": float((sums.sum(1) / counts.sum(1)).mean()),
        "lower": float(low),
        "upper": float(high),
        "status": "ok",
        "n_clusters": n,
        "repetitions": repetitions,
        "seed": seed,
    }


def paired_cluster_ci(
    y_true: np.ndarray,
    prediction_a: np.ndarray,
    prediction_b: np.ndarray,
    clusters: np.ndarray,
    mask: np.ndarray | None = None,
    repetitions: int = 1000,
    seed: int = 17,
    confidence: float = 0.95,
) -> dict:
    """CI for MAE(A)-MAE(B); negative values favor A, using a shared finite-pair mask."""
    if mask is None:
        from src.evaluation.common_mask import comparable_mask

        mask = comparable_mask(y_true, {"a": prediction_a, "b": prediction_b})
    return paired_multihorizon_ci(
        [(y_true, prediction_a, prediction_b, mask)], clusters, [1.0], repetitions, seed, confidence
    )


### `src/inference/registry.py`

Chọn family đủ 3 horizons; cheaper fallback phải qua paired CI.


In [ ]:
%%writefile src/inference/registry.py
"""Validation-only family selection with paired-CI preference for cheaper models."""

import numpy as np

from src.evaluation.bootstrap import paired_multihorizon_ci


def choose_candidate(
    families: dict,
    clusters: np.ndarray,
    order: list[str],
    repetitions: int = 1000,
    seed: int = 17,
    confidence: float = 0.95,
) -> tuple[str, list[dict]]:
    """Rank complete three-horizon families; fall back to cheaper if CI cannot distinguish."""
    horizons = (1, 6, 24)
    if not families or any(set(records) != set(horizons) for records in families.values()):
        raise ValueError("Every candidate family must provide exactly horizons 1,6,24")
    reference = next(iter(families.values()))
    for records in families.values():
        for h in horizons:
            if not np.array_equal(
                records[h][0], reference[h][0], equal_nan=True
            ) or not np.array_equal(records[h][2], reference[h][2]):
                raise ValueError("Candidate families must use identical actuals and common masks")
    baseline_names = [
        n for n in families if n in ("persistence", "seasonal_naive_24", "seasonal_naive_168")
    ]
    if not baseline_names:
        raise ValueError("Baseline fallback is mandatory")
    scales = [
        min(
            float(np.abs(families[n][h][1] - families[n][h][0])[families[n][h][2]].mean())
            for n in baseline_names
        )
        for h in horizons
    ]
    if any(s <= 0 or not np.isfinite(s) for s in scales):
        raise ValueError("Selection requires positive comparable baseline MAE")
    scores = {
        n: float(
            np.mean(
                [
                    np.abs(record[1] - record[0])[record[2]].mean() / scales[i]
                    for i, (_, record) in enumerate(sorted(records.items()))
                ]
            )
        )
        for n, records in families.items()
    }
    if any(n not in order for n in families):
        raise ValueError("All families require a predeclared complexity order")
    best = min(scores, key=lambda n: (scores[n], order.index(n)))
    rows = []
    for name in sorted(families, key=order.index):
        pairs = [
            (families[name][h][0], families[name][h][1], families[best][h][1], families[name][h][2])
            for h in horizons
        ]
        ci = paired_multihorizon_ci(pairs, clusters, scales, repetitions, seed, confidence)
        eligible = name == best or (
            order.index(name) < order.index(best) and ci["status"] == "ok" and ci["lower"] <= 0
        )
        rows.append(
            {
                "family": name,
                "normalized_mae": scores[name],
                "nominal_best": best,
                "ci_lower_vs_best": ci["lower"],
                "ci_upper_vs_best": ci["upper"],
                "ci_status": ci["status"],
                "eligible_cheaper_fallback": bool(eligible),
                "complexity_rank": order.index(name),
            }
        )
    selected = next(row["family"] for row in rows if row["eligible_cheaper_fallback"])
    for row in rows:
        row["recommended_candidate"] = row["family"] == selected
    return selected, rows


### `src/evaluation/diagnostics.py`

Quy tắc chọn beam/event và vẽ chart cùng target indices.


In [ ]:
%%writefile src/evaluation/diagnostics.py
"""Deterministic validation error atlas; selection rules are fixed, not handpicked."""

from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd

matplotlib.use("Agg")
import matplotlib.pyplot as plt


def error_atlas(
    panel,
    records: dict,
    masks: dict,
    tiers: np.ndarray,
    reference: str,
    comparison: list[str],
    reports: Path,
    fit_end: int,
) -> None:
    """Write top-5 errors, five representatives per tier, and two peaks plus one transition."""
    figures = reports / "figures"
    figures.mkdir(exist_ok=True)
    per_beam = []
    for h in (1, 6, 24):
        record = records[reference][h]
        mask = masks[h]
        count = mask.sum(0)
        per_beam.append(
            np.divide(
                np.where(mask, np.abs(record.predicted - record.actual), 0).sum(0),
                count,
                out=np.full(len(count), np.nan),
                where=count > 0,
            )
        )
    errors = np.nanmean(np.stack(per_beam), axis=0)
    valid = np.flatnonzero(np.isfinite(errors))
    top = valid[np.argsort(-errors[valid], kind="stable")[:5]]
    rows = [
        {
            "selection": "top5_reference_mean_horizon_mae",
            "beam_id": panel.beam_ids[i],
            "load_tier": tiers[i],
            "mae": errors[i],
        }
        for i in top
    ]
    # Stable, equally spaced ranks inside each fit-only load tier; no manual examples.
    fit = panel.values[: fit_end + 1, :, 0]
    count = np.isfinite(fit).sum(0)
    means = np.divide(
        np.nansum(fit, axis=0), count, out=np.full(len(count), np.nan), where=count > 0
    )
    for tier in ("Low", "Mid", "High"):
        indices = np.flatnonzero((tiers == tier) & np.isfinite(means))
        indices = indices[np.argsort(means[indices], kind="stable")]
        if len(indices):
            chosen = indices[
                np.unique(np.linspace(0, len(indices) - 1, min(5, len(indices))).astype(int))
            ]
            rows.extend(
                {
                    "selection": "five_evenly_spaced_fit_load_ranks",
                    "beam_id": panel.beam_ids[i],
                    "load_tier": tier,
                    "mae": errors[i],
                }
                for i in chosen
            )
    selections = pd.DataFrame(rows)
    selections.to_csv(reports / "error_atlas_beams.csv", index=False)
    h = 24
    record = records[reference][h]
    actual, prediction = record.actual, record.predicted
    under = actual - prediction
    finite = masks[h]
    cutoff = np.quantile(actual[finite], 0.9)
    peak = finite & (actual >= cutoff) & (under > 0)
    previous = panel.values[record.origins + h - 1, :, 0]
    transition = finite & (previous == 0) & (actual > 0) & (under > 0)
    cases, used = [], set()
    for kind, eligible in (
        ("miss_peak", peak),
        ("miss_peak", peak),
        ("zero_to_active", transition),
    ):
        positions = np.flatnonzero(eligible)
        positions = positions[np.argsort(-under.reshape(-1)[positions], kind="stable")]
        selected = next((int(p) for p in positions if int(p) not in used), None)
        if selected is None:
            continue
        used.add(selected)
        oi, bi = np.unravel_index(selected, actual.shape)
        cases.append(
            {
                "case": kind,
                "beam_id": panel.beam_ids[bi],
                "beam_index": int(bi),
                "origin_row": int(oi),
                "origin": int(record.origins[oi]),
                "target": int(record.origins[oi] + h),
                "actual": actual[oi, bi],
                "prediction": prediction[oi, bi],
                "underprediction": under[oi, bi],
            }
        )
    case_frame = pd.DataFrame(cases)
    case_frame.to_csv(reports / "error_atlas_cases.csv", index=False)
    if cases:
        fig, axes = plt.subplots(len(cases), 1, figsize=(12, 3.4 * len(cases)), squeeze=False)
        for ax, case in zip(axes[:, 0], cases):
            bi, oi = case["beam_index"], case["origin_row"]
            window = slice(max(0, oi - 24), min(len(record.origins), oi + 25))
            x = record.origins[window] + h
            ax.plot(
                x,
                np.where(finite[window, bi], actual[window, bi], np.nan),
                label="Actual",
                color="black",
                linewidth=2,
            )
            for name in comparison:
                ax.plot(
                    x,
                    np.where(finite[window, bi], records[name][h].predicted[window, bi], np.nan),
                    label=name,
                )
            ax.axvline(case["target"], color="gray", linestyle="--")
            ax.set_title(f"{case['case']} | beam {case['beam_id']} | +24h")
            ax.set_xlabel("Relative target hour")
            ax.set_ylabel("DLPRB usage")
            ax.legend()
        fig.tight_layout()
        fig.savefig(figures / "actual_vs_forecast.png", dpi=140)
        plt.close(fig)
    fig, ax = plt.subplots(figsize=(9, 4))
    for index, name in enumerate(comparison):
        values = [
            np.abs(records[name][hh].predicted - records[name][hh].actual)[masks[hh]].mean()
            for hh in (1, 6, 24)
        ]
        ax.bar(np.arange(3) + (index - 1) * 0.24, values, width=0.24, label=name)
    ax.set_xticks(np.arange(3), ["+1h", "+6h", "+24h"])
    ax.set_ylabel("Common-mask MAE")
    ax.legend()
    fig.tight_layout()
    fig.savefig(figures / "mae_by_horizon.png", dpi=140)
    plt.close(fig)
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, hh in zip(axes, (1, 6, 24)):
        for index, name in enumerate(comparison):
            values = []
            for tier in ("Low", "Mid", "High"):
                mask = masks[hh] & (tiers == tier)[None, :]
                values.append(
                    np.abs(records[name][hh].predicted - records[name][hh].actual)[mask].mean()
                    if mask.any()
                    else np.nan
                )
            ax.bar(np.arange(3) + (index - 1) * 0.24, values, width=0.24, label=name)
        ax.set_xticks(np.arange(3), ["Low", "Mid", "High"])
        ax.set_title(f"+{hh}h")
        ax.set_ylabel("Common-mask MAE")
    axes[-1].legend()
    fig.tight_layout()
    fig.savefig(figures / "mae_by_load_tier.png", dpi=140)
    plt.close(fig)
    lines = [
        "# Validation error atlas",
        "",
        f"Reference family: {reference}.",
        "Selection rules: top 5 beams by equal-weight mean per-horizon reference MAE;",
        "five equally spaced fit-load ranks per Low/Mid/High tier; two largest distinct",
        "+24h underpredictions among actual >= validation P90, plus one largest zero-to-active miss.",
        "Ties use original numeric beam/time order. No manual/cherry-picked cases.",
        "Cases are retrospective error diagnostics, not incident/congestion labels.",
        "",
    ]
    for title, frame in (("Beam selections", selections), ("Peak/transition cases", case_frame)):
        lines += [f"## {title}", "", "```csv", frame.to_csv(index=False).strip(), "```", ""]
    lines += [
        "![Actual and forecasts](figures/actual_vs_forecast.png)",
        "![Horizon MAE](figures/mae_by_horizon.png)",
        "![Load-tier MAE](figures/mae_by_load_tier.png)",
    ]
    (reports / "error_atlas.md").write_text("\n".join(lines), encoding="utf-8")


### `scripts/quality_audit.py`

Chỉ đọc artifact metadata/weights để xác minh schema, seed và preprocessing.


In [ ]:
%%writefile scripts/quality_audit.py
"""Audit current artifacts against Week 5 causal/preprocessing contracts without fitting."""

import json
from pathlib import Path

import numpy as np
import torch

from src.evaluation.origins import evaluation_origins, train_origins
from src.evaluation.validation import validation_inputs
from src.features.lightgbm_features import FEATURE_NAMES
from src.features.gru_inputs import GRUPreprocessor, SCHEMA
from src.models.lgbm import load_lgbm
from src.data.io import sha256
from src.utils.config import load_config


def audit(data_path: str = "configs/data.yaml", write: bool = True) -> dict:
    """Require all rerun artifacts, exact schema, provenance, origin counts and preprocessing parity."""
    data, lgb_config, gru_config = (
        load_config(p) for p in (data_path, "configs/lightgbm.yaml", "configs/gru.yaml")
    )
    panel, split = validation_inputs(data)
    checks, digests, preprocessors, samples = [], {}, [], []

    def check(label: str, condition: bool, detail: str = "") -> None:
        checks.append(
            {
                "check": label.replace("\\", "/"),
                "status": "PASS" if condition else "FAIL",
                "detail": detail,
            }
        )

    check(
        "feature_schema",
        len(FEATURE_NAMES) == 40 and not any("beam_fit" in n for n in FEATURE_NAMES),
    )
    for config_name in lgb_config["configs"]:
        for h in split["horizons"]:
            directory = Path(lgb_config["artifact_dir"]) / config_name / f"h{h}"
            try:
                model = load_lgbm(directory)
                meta = model.metadata
                check(
                    str(directory),
                    meta["dataset_revision"] == data["revision"]
                    and meta["config"] == lgb_config["configs"][config_name]
                    and meta["seed"] == lgb_config["seed"]
                    and bool(meta["git_commit"])
                    and meta["fit_end"] == split["fit"][1]
                    and meta["feature_names"] == FEATURE_NAMES,
                )
                for name, origins, count_key, missing_key in (
                    (
                        "train",
                        train_origins(split["fit"][1], h, stride=lgb_config["training_stride"]),
                        "train_rows",
                        "train_missing_targets",
                    ),
                    (
                        "validation",
                        evaluation_origins(split["fit"][1], split["validation"][1], h),
                        "validation_rows",
                        "validation_missing_targets",
                    ),
                ):
                    expected = np.isfinite(panel.values[origins + h, :, 0]).sum()
                    check(
                        f"{directory}/{name}_samples",
                        meta[count_key] == int(expected)
                        and meta[missing_key] == len(origins) * len(panel.beam_ids) - int(expected),
                    )
                for filename in ("model.txt", "meta.json", "context.npz"):
                    digests[(directory / filename).as_posix()] = sha256(directory / filename)
            except (OSError, ValueError, KeyError) as exc:
                check(str(directory), False, str(exc))
    for variant in gru_config["variants"]:
        for seed in gru_config["seeds"]:
            directory = Path(gru_config["artifact_dir"]) / variant / f"seed{seed}"
            try:
                meta = json.loads((directory / "meta.json").read_text())
                prep = GRUPreprocessor.load(directory / "preprocessor.npz")
                state = torch.load(directory / "best.pt", map_location="cpu", weights_only=True)
                k = 1 if variant == "A" else 4
                check(
                    str(directory),
                    meta.get("preprocessing_schema") == SCHEMA
                    and meta["dataset_revision"] == data["revision"]
                    and bool(meta["git_commit"])
                    and meta["seed"] == seed
                    and state["identity"]["seed"] == seed
                    and state["identity"]["run_identity"]["variant"] == variant
                    and prep.variant == variant
                    and prep.fit_end == split["fit"][1]
                    and meta["phase_features"] == 4
                    and state["model"]["head.0.weight"].shape[1]
                    == meta["architecture"]["hidden_size"] + 4
                    and state["model"]["gru.weight_ih_l0"].shape[1] == 2 * k
                    and all(getattr(prep, p).shape == (k,) for p in ("median", "mean", "scale")),
                )
                identity = state["identity"]["run_identity"]
                samples.append(identity["beam_sample"])
                preprocessors.append((prep.median[0], prep.mean[0], prep.scale[0]))
                from src.evaluation.origins import train_origins_joint

                expected_train = len(
                    train_origins_joint(split["fit"][1], stride=gru_config["training_stride"])
                ) * len(samples[-1])
                expected_val = len(
                    evaluation_origins(split["fit"][1], split["validation"][1], 24)
                ) * len(panel.beam_ids)
                check(
                    f"{directory}/samples",
                    meta["train_samples"] == expected_train
                    and meta["validation_joint_samples"] == expected_val,
                )
                for filename in (
                    "best.pt",
                    "last.pt",
                    "preprocessor.npz",
                    "meta.json",
                    "history.jsonl",
                ):
                    digests[(directory / filename).as_posix()] = sha256(directory / filename)
            except (OSError, ValueError, KeyError, RuntimeError) as exc:
                check(str(directory), False, str(exc))
    check(
        "shared_sample_and_DLPRB_statistics",
        bool(samples)
        and all(x == samples[0] for x in samples)
        and all(x == preprocessors[0] for x in preprocessors),
    )
    status = "PASS" if checks and all(row["status"] == "PASS" for row in checks) else "FAIL"
    result = {
        "status": status,
        "checks": checks,
        "artifact_sha256": digests,
        "dataset_revision": data["revision"],
        "holdout_evaluated": False,
        "p1": "skipped: legacy schemas required retraining",
    }
    if write:
        reports = Path(data["reports_dir"])
        (reports / "model_quality_gate.json").write_text(json.dumps(result, indent=2), encoding="utf-8", newline="\n")
        lines = [
            "# Week 5 model quality gate",
            "",
            f"Status: **{status}**.",
            "Legacy W3/W4 artifacts FAILED the updated plan (44/static6); all were retrained on cloud.",
            "No inference-only patch was used to approve legacy artifacts. P1 quantiles skipped.",
            "Normalization is fitted once at the cutoff, globally per KPI; earlier fit origins are",
            "training samples, not separate deployed episodes. No holdout has been evaluated.",
            "",
        ]
        lines += [f"- {r['status']}: {r['check']} {r['detail']}" for r in checks]
        lines += [
            "",
            "Hashes and provenance: model_quality_gate.json; initial failures: model_quality_gate_initial.json.",
        ]
        (reports / "model_quality_gate.md").write_text("\n".join(lines), encoding="utf-8")
    return result


if __name__ == "__main__":
    result = audit()
    print(result["status"])
    if result["status"] != "PASS":
        raise SystemExit(1)


### Ví dụ debug common mask (không training)

Actual vẫn hữu hạn nhưng một candidate thiếu forecast: cặp đó bị loại khỏi tất cả,
không được báo nhầm là actual bị missing. History NaN không nằm trong điều kiện mask này.


In [ ]:
from src.evaluation.common_mask import comparable_mask, coverage_report
actual=np.array([[0.,1.,2.],[3.,4.,5.]])
forecast_a=actual.copy()
forecast_b=actual+0.5
forecast_b[0,1]=np.nan
mask=comparable_mask(actual,{'a':forecast_a,'b':forecast_b})
print('Common mask:')
print(mask)
print(coverage_report(actual,forecast_a,mask))
